# InferForge — NVIDIA GPU Hardware Validation (Google Colab)

**Project:** InferForge Project 01 — GPU System Inspector (`inferforge-gpu`)  
**Purpose:** Produce reproducible NVIDIA hardware-validation evidence using the project's own CLI, hardware tests and evidence script.  
**Status:** This notebook is a permanent Project 01 asset. It does **not** by itself mean hardware validation has been completed.

## What InferForge is

InferForge is a modular multi-project repository. Project 01 is a **read-only** GPU environment inspector: it collects system, NVIDIA/NVML, driver, CUDA Toolkit, PyTorch and runtime facts, then explains them with evidence-based diagnostics (D01–D08) and optional free-VRAM readiness (Mode B).

## What this notebook validates

On a Colab **GPU** runtime, this notebook:

1. Confirms an NVIDIA GPU is visible (`nvidia-smi`).
2. Installs InferForge **from source** (never from a PyPI package).
3. Runs the production CLI (`inferforge-gpu inspect`).
4. Runs the project's hardware tests (`pytest -m hardware`).
5. Generates the official evidence bundle via `validation/hardware_evidence.py`.
6. Packages the evidence for download and review.

## What it does **not** validate

- That CUDA kernels will run for your training workload (ordinary inspection never allocates tensors or launches kernels).
- Bare-metal PCI behaviour beyond what Colab's VM/container exposes (HW-02 on Colab is "native Linux in a VM/container", not bare metal).
- Multi-GPU ordering (HW-09 — Colab usually has one GPU).
- Windows WDDM (HW-08), WSL2-with-NVIDIA (HW-07), or macOS (HW-11).
- Publication readiness or PyPI packaging.

## Why an NVIDIA GPU runtime is required

Hardware items HW-01, HW-03, HW-04, HW-05 and HW-10 need a real NVIDIA driver interface. A CPU-only Colab runtime can only demonstrate graceful degradation (like HW-13). **This notebook stops with an explicit `BLOCKED` result if no NVIDIA GPU is available.** CPU-only execution is never presented as NVIDIA hardware validation.

## How the evidence is used

The downloaded archive (especially `SUMMARY.md`, `summary.json`, `hardware-tests.xml`) is attached to a GitHub issue or sent to the project owner. The owner records outcomes in `docs/app_tracker.md` §6. Only a **passed** hardware test with evidence may move an item to `HARDWARE_VALIDATED`. A skipped test is **not** a pass. The evidence script's verdict is at most `EVIDENCE_COLLECTED` pending owner review — never self-declared `HARDWARE_VALIDATED`.

## Contributor workflow (GitHub-first)

1. Open this notebook from the public branch `feat/gpu-system-inspector-baseline` (Colab “Open from GitHub”, or upload the `.ipynb`).
2. Select a **GPU** runtime.
3. Keep the defaults: `SOURCE_MODE = "git"` and `GIT_BRANCH = "feat/gpu-system-inspector-baseline"` (Project 01 is on that public branch).
4. Run cells in order — the notebook downloads the public source archive itself (no manual `.tar.gz` upload).
5. Download the evidence bundle and attach it to a GitHub issue.

If the GitHub download fails (network, branch rename, or a private fork), use `SOURCE_MODE = "archive"` and upload a project tarball (see Section 3). Do not silently switch modes without recording why.

> **Warning — Colab GPU availability.** Google Colab GPU runtimes depend on your account type, quotas and current capacity. A GPU may be unavailable, downgraded or disconnected mid-session. If that happens, stop and re-run later; do not treat a CPU fallback as hardware validation.



## Configuration (edit before running)

Set how you obtain the InferForge source, then run the remaining cells **in order**.

| Setting | Meaning |
| ------- | ------- |
| `SOURCE_MODE = "git"` | **Recommended default.** Download the public GitHub source archive for `GIT_BRANCH` (no login, no manual tarball). |
| `SOURCE_MODE = "archive"` | Fallback if the GitHub download fails, or for offline / private trees. You upload a project tarball yourself. |
| `GIT_BRANCH` | Public branch to fetch. Default: `feat/gpu-system-inspector-baseline` (contains Project 01). |
| `GIT_REF` | Optional commit SHA. When set, that exact commit is downloaded instead of the branch tip. |
| `OPT_IN_CUDA_SMOKE` | Opt-in: tiny CUDA compute in a **separate** child after inspection (not part of ordinary inspection). |
| `OPT_IN_HW12_ALLOCATION` | Opt-in: HW-12 allocates ~75% of free GPU memory in a child process. |

**Do not paste GitHub personal access tokens into this notebook.** Public downloads need no credentials.

**Public source:** Project 01 is on `https://github.com/nawalkumar/InferForge` branch `feat/gpu-system-inspector-baseline`. The configuration cell already defaults to that branch. Change `GIT_BRANCH` / `GIT_REF` only if you intentionally validate a different public ref.


In [ ]:
# === USER CONFIGURATION =====================================================
from pathlib import Path

# Recommended default: public GitHub download (no manual .tar.gz).
# Falls back to "archive" for unpublished / offline / private workflows.
SOURCE_MODE = "git"

# --- Workflow A (git): public GitHub, no authentication ---
REPO_OWNER = "nawalkumar"
REPO_NAME = "InferForge"
REPO_URL = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
# Public GitHub branch containing Project 01.
# Keep this branch aligned with the documented Colab validation workflow.
GIT_BRANCH = "feat/gpu-system-inspector-baseline"
GIT_REF = None  # optional full commit SHA; when set, that commit is fetched instead of the branch tip

# --- Workflow B (archive): manual upload ---
# Build on a machine that has the current Project 01 tree, then upload via Colab Files:
#   cd projects && tar czf /tmp/gsi-project.tar.gz \
#     --exclude=__pycache__ --exclude=.pytest_cache --exclude=.coverage --exclude='*.pyc' \
#     01-gpu-system-inspector
ARCHIVE_PATH = Path("/content/gsi-project.tar.gz")

# Evidence
EVIDENCE_LABEL = "Colab"  # refined later with the GPU model
EVIDENCE_ROOT = Path("/content/gsi-evidence")
REQUIRED_VRAM_GIB = "1"  # Mode B free-memory check (GiB); does not prove a model fits

# Optional checks — OFF by default (allocate / execute CUDA outside ordinary inspection)
OPT_IN_CUDA_SMOKE = False
OPT_IN_HW12_ALLOCATION = False

# Working directories (all downloads/extracts stay under here)
WORK_ROOT = Path("/content/inferforge-validation")
print("Configuration loaded.")
print(f"  SOURCE_MODE={SOURCE_MODE!r}")
print(f"  REPO_URL={REPO_URL}")
print(f"  GIT_BRANCH={GIT_BRANCH!r}  GIT_REF={GIT_REF!r}")
print(f"  OPT_IN_CUDA_SMOKE={OPT_IN_CUDA_SMOKE}  OPT_IN_HW12_ALLOCATION={OPT_IN_HW12_ALLOCATION}")


## Section 2 — Runtime verification

Confirm Python, OS and **NVIDIA GPU visibility** before any installation. If this cell reports `BLOCKED`, do not continue the hardware-validation path.



In [ ]:
import os
import platform
import shutil
import subprocess
import sys
from datetime import datetime, timezone

RUNTIME = {
    "collected_at": datetime.now(timezone.utc).isoformat(),
    "python": sys.version,
    "python_version": platform.python_version(),
    "platform": platform.platform(),
    "machine": platform.machine(),
    "in_colab": False,
    "gpu_visible": False,
    "gpu_count": 0,
    "gpu_name": None,
    "driver_version": None,
    "nvidia_smi": None,
    "cuda_env": {},
    "verdict": None,
}

try:
    import google.colab  # noqa: F401
    RUNTIME["in_colab"] = True
except Exception:
    RUNTIME["in_colab"] = False

print("Python:", RUNTIME["python_version"], f"({platform.python_implementation()})")
print("Platform:", RUNTIME["platform"])
print("Machine:", RUNTIME["machine"])
print("Google Colab:", RUNTIME["in_colab"])

if tuple(map(int, platform.python_version_tuple()[:2])) < (3, 11):
    raise SystemExit(
        "BLOCKED: InferForge Project 01 requires Python >= 3.11. "
        f"This runtime has {platform.python_version()}."
    )

nvidia_smi = shutil.which("nvidia-smi")
if nvidia_smi is None and Path("/usr/lib/wsl/lib/nvidia-smi").exists():
    nvidia_smi = "/usr/lib/wsl/lib/nvidia-smi"
RUNTIME["nvidia_smi"] = nvidia_smi

if nvidia_smi is None:
    RUNTIME["verdict"] = "BLOCKED: nvidia-smi not found — no NVIDIA driver interface visible"
    print("\n" + RUNTIME["verdict"])
    print("Select Runtime → Change runtime type → GPU, then Runtime → Restart session, and re-run.")
    raise SystemExit(RUNTIME["verdict"])

header = subprocess.run([nvidia_smi], capture_output=True, text=True, timeout=60)
print("\n--- nvidia-smi ---")
print(header.stdout if header.returncode == 0 else header.stderr)
if header.returncode != 0:
    RUNTIME["verdict"] = f"BLOCKED: nvidia-smi failed (exit {header.returncode})"
    raise SystemExit(RUNTIME["verdict"])

query = subprocess.run(
    [
        nvidia_smi,
        "--query-gpu=index,name,driver_version,memory.total",
        "--format=csv,noheader",
    ],
    capture_output=True,
    text=True,
    timeout=60,
)
rows = [r.strip() for r in query.stdout.splitlines() if r.strip()]
RUNTIME["gpu_count"] = len(rows)
RUNTIME["gpu_visible"] = len(rows) > 0
if rows:
    parts = [c.strip() for c in rows[0].split(",")]
    RUNTIME["gpu_name"] = parts[1] if len(parts) > 1 else rows[0]
    RUNTIME["driver_version"] = parts[2] if len(parts) > 2 else None

print("GPU count:", RUNTIME["gpu_count"])
print("Primary GPU:", RUNTIME["gpu_name"])
print("Driver:", RUNTIME["driver_version"])
for row in rows:
    print(" ", row)

for key in ("CUDA_HOME", "CUDA_PATH", "LD_LIBRARY_PATH", "NVIDIA_VISIBLE_DEVICES"):
    RUNTIME["cuda_env"][key] = os.environ.get(key)
print("\nCUDA-related environment (may be unset):")
for k, v in RUNTIME["cuda_env"].items():
    print(f"  {k}={v!r}")

if not RUNTIME["gpu_visible"]:
    RUNTIME["verdict"] = "BLOCKED: nvidia-smi ran but reported zero GPUs"
    raise SystemExit(RUNTIME["verdict"])

RUNTIME["verdict"] = "NVIDIA GPU visible — continue"
print("\nRuntime check:", RUNTIME["verdict"])
# Refine the evidence label with the GPU model when possible.
if "EVIDENCE_LABEL" in globals() and RUNTIME["gpu_name"]:
    EVIDENCE_LABEL = f"Colab {RUNTIME['gpu_name']}"
    print("Evidence label:", EVIDENCE_LABEL)


## Section 3 — Obtain InferForge source

The notebook records **source mode**, **repository URL**, **branch**, and **commit SHA** (when available) so a run can be reproduced. Missing metadata is labelled `unknown` — never invented.

### Workflow A — GitHub-first (recommended, no manual archive)

With `SOURCE_MODE = "git"` the notebook:

1. Resolves the commit for `GIT_BRANCH` (or uses `GIT_REF` when set) via the public GitHub API when possible.
2. Downloads the public source archive over HTTPS (no login).
3. Extracts it only under `/content/inferforge-validation/`.
4. Uses `projects/01-gpu-system-inspector/` from that archive.
5. Verifies required files before install.

You do **not** need to download or upload a `.tar.gz` yourself in this mode.

```text
https://github.com/nawalkumar/InferForge/archive/refs/heads/<GIT_BRANCH>.tar.gz
# or, when GIT_REF is set:
https://github.com/nawalkumar/InferForge/archive/<GIT_REF>.tar.gz
```

No GitHub credentials are required. **Do not paste personal access tokens into cells.**

> **Default public branch:** Workflow A targets `feat/gpu-system-inspector-baseline` on `nawalkumar/InferForge` (see the configuration cell). `main` is a different branch and is **not** required for this notebook. If Workflow A fails, record the error and use Workflow B — do not silently change the branch to make the run pass.

### Workflow B — Archive fallback (GitHub download failed / offline / private)

Use when HTTPS archive download fails, you are offline, or you need a private tree:

1. On a machine with the current Project 01 checkout (repository `projects/` directory):

```bash
cd projects && tar czf /tmp/gsi-project.tar.gz \
  --exclude=__pycache__ --exclude=.pytest_cache --exclude=.coverage \
  --exclude='*.pyc' 01-gpu-system-inspector
```

2. In Colab, open the **Files** pane and upload `gsi-project.tar.gz` to `/content/gsi-project.tar.gz` (or set `ARCHIVE_PATH`).
3. Set `SOURCE_MODE = "archive"` in the configuration cell and re-run from Section 3.

The notebook never silently falls back from git to archive, and never installs InferForge from PyPI.


In [ ]:
import hashlib
import json
import os
import shutil
import tarfile
import urllib.error
import urllib.request
from pathlib import Path

EXPECTED_FILES = (
    "pyproject.toml",
    "src/gpu_system_inspector/__init__.py",
    "validation/hardware_evidence.py",
    "tests/hardware/test_nvidia_hardware.py",
    "docs/hardware_validation.md",
)
OPTIONAL_FILES = ("notebooks/inferforge_nvidia_gpu_validation.ipynb",)
PROJECT_REL = Path("projects") / "01-gpu-system-inspector"
USER_AGENT = "inferforge-colab-validation/1.0"

SOURCE = {
    "mode": SOURCE_MODE,
    "repository": f"{REPO_OWNER}/{REPO_NAME}",
    "repo_url": REPO_URL,
    "branch": None,
    "commit": None,
    "commit_status": "unknown",
    "download_url": None,
    "archive_path": None,
    "archive_sha256": None,
    "archive_bytes": None,
    "project_root": None,
    "error": None,
}

WORK_ROOT.mkdir(parents=True, exist_ok=True)


def require_project(root: Path) -> Path:
    missing = [rel for rel in EXPECTED_FILES if not (root / rel).is_file()]
    if missing:
        raise SystemExit(
            "Source check FAILED — expected Project 01 files are missing:\n  - "
            + "\n  - ".join(missing)
            + "\n\nThe downloaded/uploaded tree is not a usable Project 01 checkout.\n"
            "If SOURCE_MODE='git': the chosen public branch does not contain Project 01 yet "
            f"(tried branch={SOURCE.get('branch')!r}, commit={SOURCE.get('commit')!r}).\n"
            "Set GIT_BRANCH to a published branch that contains projects/01-gpu-system-inspector, "
            "or switch to SOURCE_MODE='archive' and upload a current project tarball.\n"
            "The notebook will not fall back to PyPI or to a stale tree."
        )
    for rel in OPTIONAL_FILES:
        if not (root / rel).is_file():
            print(f"Note: optional file missing: {rel}")
    return root


def safe_extract(archive: Path, dest: Path) -> None:
    dest.mkdir(parents=True, exist_ok=True)
    dest_resolved = dest.resolve()
    with tarfile.open(archive, "r:*") as tf:
        for member in tf.getmembers():
            name = member.name
            if name.startswith("/") or ".." in Path(name).parts:
                raise SystemExit(f"Refusing unsafe archive member: {name!r}")
            target = (dest / name).resolve()
            if dest_resolved not in target.parents and target != dest_resolved:
                raise SystemExit(f"Refusing extract outside workspace: {name!r}")
        try:
            tf.extractall(dest, filter="data")
        except TypeError:
            tf.extractall(dest)


def find_project(extract_root: Path) -> Path | None:
    """Locate Project 01 under an extracted archive.

    Prefer ``projects/01-gpu-system-inspector`` (GitHub monorepo layout). Do **not**
    treat a workspace root that has ``projects/`` but no Project 01 as a hit — that
    would confuse a published monorepo tip that still lacks this project with a
    valid flat project tree.
    """
    direct = extract_root / PROJECT_REL
    if direct.is_dir():
        return direct
    # GitHub archives extract to <repo>-<ref>/...
    for child in sorted(extract_root.iterdir()):
        if not child.is_dir():
            continue
        candidate = child / PROJECT_REL
        if candidate.is_dir():
            return candidate
        # Monorepo tip without Project 01 → explicit miss (caller fails clearly).
        if (child / "projects").is_dir() and not candidate.is_dir():
            return None
    if (extract_root / "projects").is_dir() and not direct.is_dir():
        return None
    # Project-only archive layouts (SOURCE_MODE=archive)
    for candidate in (
        extract_root / "01-gpu-system-inspector",
        extract_root / PROJECT_REL,
    ):
        if candidate.is_dir():
            return candidate
    # Flat tree: project root itself (must look like Project 01, not only a workspace).
    def _looks_like_project(root: Path) -> bool:
        return (root / "pyproject.toml").is_file() and (
            root / "src" / "gpu_system_inspector" / "__init__.py"
        ).is_file()

    if _looks_like_project(extract_root):
        return extract_root
    for child in sorted(extract_root.iterdir()):
        if child.is_dir() and _looks_like_project(child):
            return child
    return None


def http_json(url: str) -> tuple[dict | None, str | None]:
    req = urllib.request.Request(
        url,
        headers={"Accept": "application/vnd.github+json", "User-Agent": USER_AGENT},
    )
    try:
        with urllib.request.urlopen(req, timeout=60) as resp:
            return json.load(resp), None
    except urllib.error.HTTPError as exc:
        body = exc.read().decode("utf-8", errors="replace")[:300]
        return None, f"HTTP {exc.code} for {url}: {body}"
    except Exception as exc:  # noqa: BLE001 — surface any network/DNS failure clearly
        return None, f"{type(exc).__name__}: {exc}"


def http_download(url: str, dest: Path) -> None:
    req = urllib.request.Request(url, headers={"User-Agent": USER_AGENT})
    try:
        with urllib.request.urlopen(req, timeout=300) as resp, dest.open("wb") as out:
            shutil.copyfileobj(resp, out)
    except urllib.error.HTTPError as exc:
        raise SystemExit(
            f"Download failed: HTTP {exc.code} for {url}\n"
            "Check that REPO_OWNER/REPO_NAME/GIT_BRANCH (or GIT_REF) refer to a public "
            "repository and ref. The notebook will not fall back to another source or PyPI.\n"
            "For unpublished work, set SOURCE_MODE='archive' and upload a project tarball."
        ) from exc
    except Exception as exc:  # noqa: BLE001
        raise SystemExit(f"Download failed for {url}: {type(exc).__name__}: {exc}") from exc


if SOURCE_MODE == "git":
    SOURCE["branch"] = GIT_BRANCH
    ref = GIT_REF or GIT_BRANCH
    if not ref:
        raise SystemExit("SOURCE_MODE='git' requires GIT_BRANCH or GIT_REF.")

    # Resolve commit SHA when the public API allows; otherwise leave unknown.
    api_url = f"https://api.github.com/repos/{REPO_OWNER}/{REPO_NAME}/commits/{ref}"
    print("Resolving commit via", api_url)
    payload, api_error = http_json(api_url)
    if payload and payload.get("sha"):
        SOURCE["commit"] = payload["sha"]
        SOURCE["commit_status"] = "resolved"
        print("Commit SHA:", SOURCE["commit"])
    else:
        SOURCE["commit"] = None
        SOURCE["commit_status"] = "unknown"
        SOURCE["error"] = api_error
        print("Commit SHA: unknown (" + (api_error or "no sha in response") + ")")

    if GIT_REF:
        download_url = f"https://github.com/{REPO_OWNER}/{REPO_NAME}/archive/{GIT_REF}.tar.gz"
    else:
        download_url = (
            f"https://github.com/{REPO_OWNER}/{REPO_NAME}/archive/refs/heads/"
            f"{GIT_BRANCH}.tar.gz"
        )
    SOURCE["download_url"] = download_url
    download_dir = WORK_ROOT / "github-download"
    if download_dir.exists():
        shutil.rmtree(download_dir)
    download_dir.mkdir(parents=True)
    archive_path = download_dir / "source.tar.gz"
    print("Downloading", download_url)
    http_download(download_url, archive_path)
    digest = hashlib.sha256(archive_path.read_bytes()).hexdigest()
    SOURCE["archive_path"] = str(archive_path)
    SOURCE["archive_sha256"] = digest
    SOURCE["archive_bytes"] = archive_path.stat().st_size
    print(f"Downloaded {SOURCE['archive_bytes']} bytes, sha256={digest[:12]}…")

    extract_root = download_dir / "extract"
    safe_extract(archive_path, extract_root)
    project = find_project(extract_root)
    if project is None:
        top = [p.name for p in extract_root.iterdir()] if extract_root.exists() else []
        raise SystemExit(
            "GitHub archive extracted, but projects/01-gpu-system-inspector was not found.\n"
            f"branch={GIT_BRANCH!r} commit={SOURCE['commit']!r}\n"
            f"Top-level entries: {top}\n"
            "This public ref does not contain Project 01. Set GIT_BRANCH to a published "
            "branch that does, or use SOURCE_MODE='archive'."
        )
    # Ensure the project path stays inside WORK_ROOT.
    project_resolved = project.resolve()
    if WORK_ROOT.resolve() not in project_resolved.parents and project_resolved != WORK_ROOT.resolve():
        raise SystemExit(f"Refusing project path outside WORK_ROOT: {project}")
    SOURCE["project_root"] = str(require_project(project))
    print("Project root:", SOURCE["project_root"])
    print("Source mode=git repository=", SOURCE["repository"], "branch=", SOURCE["branch"])

elif SOURCE_MODE == "archive":
    archive = Path(ARCHIVE_PATH)
    if not archive.is_file():
        raise SystemExit(
            f"Archive not found: {archive}\n"
            "Upload gsi-project.tar.gz via the Colab Files pane, or update ARCHIVE_PATH.\n"
            "Build it with:\n"
            "  cd projects && tar czf /tmp/gsi-project.tar.gz "
            "--exclude=__pycache__ --exclude=.pytest_cache --exclude=.coverage "
            "--exclude='*.pyc' 01-gpu-system-inspector"
        )
    digest = hashlib.sha256(archive.read_bytes()).hexdigest()
    SOURCE["archive_path"] = str(archive)
    SOURCE["archive_sha256"] = digest
    SOURCE["archive_bytes"] = archive.stat().st_size
    SOURCE["branch"] = None
    SOURCE["commit"] = None
    SOURCE["commit_status"] = "not_applicable"
    extract_root = WORK_ROOT / "from-archive"
    if extract_root.exists():
        shutil.rmtree(extract_root)
    print(f"Extracting {archive} ({SOURCE['archive_bytes']} bytes, sha256={digest[:12]}…)")
    safe_extract(archive, extract_root)
    project = find_project(extract_root)
    if project is None:
        raise SystemExit(
            f"Could not find 01-gpu-system-inspector under {extract_root}. "
            f"Top-level entries: {[p.name for p in extract_root.iterdir()]}"
        )
    project_resolved = project.resolve()
    if WORK_ROOT.resolve() not in project_resolved.parents and project_resolved != WORK_ROOT.resolve():
        raise SystemExit(f"Refusing project path outside WORK_ROOT: {project}")
    SOURCE["project_root"] = str(require_project(project))
    print("Archive sha256:", digest)
    print("Project root:", SOURCE["project_root"])
else:
    raise SystemExit(f"Unknown SOURCE_MODE={SOURCE_MODE!r}; use 'git' or 'archive'.")

PROJECT_ROOT = Path(SOURCE["project_root"])
os.chdir(PROJECT_ROOT)
print("cwd:", Path.cwd())
print("Install will use this local tree only (pip install .), never PyPI.")


## Section 4 — Install InferForge from source

Install the checked-out project into **this** Python environment (Colab's runtime), so the PyTorch probe inspects Colab's own PyTorch build.

- Distribution: `01-gpu-system-inspector` (not published; classifier `Private :: Do Not Upload`).
- Import package: `gpu_system_inspector`.
- Console script: `inferforge-gpu`.
- Runtime dependency: `nvidia-ml-py` (declared in `pyproject.toml`).
- Test tools for this notebook: `pytest`, `jsonschema`.

PyTorch is **not** installed by this project (runtime-detected). Colab GPU images normally already include a CUDA-enabled PyTorch.

```bash
python -m pip install -q . pytest jsonschema
```



In [ ]:
import importlib
import importlib.metadata
import shutil
import subprocess
import sys
from pathlib import Path

assert Path.cwd().resolve() == PROJECT_ROOT.resolve(), (
    f"Expected to be in {PROJECT_ROOT}, cwd is {Path.cwd()}"
)

print("Installing from:", PROJECT_ROOT)
install = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", ".", "pytest", "jsonschema"],
    capture_output=True,
    text=True,
    timeout=600,
)
if install.returncode != 0:
    print(install.stdout)
    print(install.stderr)
    raise SystemExit(
        "pip install failed. Common causes:\n"
        "- wrong directory (must be projects/01-gpu-system-inspector)\n"
        "- Python < 3.11\n"
        "- conflicting old `pynvml` package (note it; the report records which binding was imported)\n"
        f"exit code: {install.returncode}"
    )
print("pip install: OK")

# Fresh import after install
for name in list(sys.modules):
    if name == "gpu_system_inspector" or name.startswith("gpu_system_inspector."):
        del sys.modules[name]

import gpu_system_inspector as gsi

INSTALL = {
    "distribution": "01-gpu-system-inspector",
    "version": gsi.__version__,
    "module_file": gsi.__file__,
    "console_script": shutil.which("inferforge-gpu"),
    "nvidia_ml_py": None,
    "torch": None,
}
try:
    INSTALL["nvidia_ml_py"] = importlib.metadata.version("nvidia-ml-py")
except importlib.metadata.PackageNotFoundError:
    INSTALL["nvidia_ml_py"] = "NOT INSTALLED"
try:
    INSTALL["torch"] = importlib.metadata.version("torch")
except importlib.metadata.PackageNotFoundError:
    INSTALL["torch"] = "not installed"

print("Package version:", INSTALL["version"])
print("Imported from:", INSTALL["module_file"])
print("inferforge-gpu on PATH:", INSTALL["console_script"])
print("nvidia-ml-py:", INSTALL["nvidia_ml_py"])
print("torch (optional):", INSTALL["torch"])

version = subprocess.run(
    [sys.executable, "-m", "gpu_system_inspector", "version"],
    capture_output=True,
    text=True,
    timeout=60,
)
print("\n--- inferforge-gpu version ---")
print(version.stdout)
if version.returncode != 0:
    raise SystemExit(f"version command failed: {version.stderr}")
if INSTALL["console_script"] is None:
    print(
        "Note: console script not on PATH; using `python -m gpu_system_inspector` "
        "(equivalent entry point)."
    )


## Section 5 — Live inspection (production CLI)

Runs the real inspector end to end. Capture text, JSON, exit code and key observations.

A Mode A exit code of `0` means **a report was produced**, not that every hardware capability works. Missing layers appear as statuses with reasons (`NOT_DETECTED`, `NOT_SUPPORTED`, …), never as fabricated zeros.

Schema validation uses the **shipped** JSON Schema from the installed package.



In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import jsonschema
from gpu_system_inspector.reporting import load_report_schema, parse_report_json

LIVE_DIR = WORK_ROOT / "live-inspect"
LIVE_DIR.mkdir(parents=True, exist_ok=True)
CLI = [sys.executable, "-m", "gpu_system_inspector"]

def run_inspect(label: str, args: list[str], json_path: Path | None = None):
    cmd = [*CLI, "inspect", *args]
    if json_path is not None:
        cmd += ["--json", str(json_path)]
    print("\n$", " ".join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
    text_path = LIVE_DIR / f"{label}.txt"
    err_path = LIVE_DIR / f"{label}.stderr.txt"
    text_path.write_text(result.stdout)
    err_path.write_text(result.stderr)
    print(f"exit={result.returncode}  stdout→{text_path.name}  stderr→{err_path.name}")
    if result.stderr.strip():
        print("stderr:", result.stderr.strip()[:500])
    return result

# Mode A — human-readable + JSON
json_a = LIVE_DIR / "inspect-modeA.json"
mode_a = run_inspect("inspect-modeA", [], json_a)
print(mode_a.stdout[:2500])
if len(mode_a.stdout) > 2500:
    print("… [truncated; full text in", LIVE_DIR / "inspect-modeA.txt", "]")

# Mode A redacted JSON (safe to share more widely)
json_redacted = LIVE_DIR / "inspect-modeA-redacted.json"
mode_a_redacted = run_inspect("inspect-modeA-redacted", ["--redact"], json_redacted)

# Mode B readiness
json_b = LIVE_DIR / "inspect-modeB.json"
mode_b = run_inspect("inspect-modeB", ["--required-vram", str(REQUIRED_VRAM_GIB)], json_b)

# Schema validation
report = json.loads(json_a.read_text())
jsonschema.validate(report, load_report_schema())
parsed = parse_report_json(json_a.read_text())
assert parsed.data_origin.value == "LIVE"

gpu = report["gpu"]
driver = report["driver"]
pytorch = report["frameworks"]["pytorch"]
print("\n=== Key observations ===")
print("data_origin:", report["data_origin"])
print("device_count:", gpu["device_count"])
if gpu.get("devices"):
    for d in gpu["devices"]:
        print(
            f"  {d['device_ref']}: name={d['name']['value']!r} "
            f"uuid_status={d['uuid']['status']} pci={d['pci_bus_id']['status']}"
        )
print("driver_version:", driver["driver_version"])
print("driver_supported_cuda:", driver["driver_supported_cuda_version"])
print("toolkits:", report["cuda_environment"]["toolkits"]["status"])
print("pytorch status:", pytorch["status"], "cuda_available:", pytorch["cuda_available"])
if pytorch.get("devices"):
    for d in pytorch["devices"]:
        print(
            f"  cuda:{d['torch_index']} → correlated_device_ref "
            f"{d['correlated_device_ref']['status']}={d['correlated_device_ref']['value']!r}"
        )
print("Mode A exit:", mode_a.returncode, "  Mode B exit:", mode_b.returncode)
report_b = json.loads(json_b.read_text())
jsonschema.validate(report_b, load_report_schema())
if report_b.get("readiness"):
    print(
        "readiness:",
        report_b["readiness"]["overall_status"],
        report_b["readiness"]["overall_reason"],
    )
else:
    print("readiness: null (unexpected for Mode B)")
print("findings (Mode A):")
for f in report["diagnostics"]:
    print(f"  [{f['status']}] {f['check_id']} {f['title']}  confidence={f['confidence']}")

LIVE = {
    "mode_a_exit": mode_a.returncode,
    "mode_b_exit": mode_b.returncode,
    "schema": "valid",
    "data_origin": report["data_origin"],
    "device_count": gpu["device_count"],
    "json_a": str(json_a),
    "json_redacted": str(json_redacted),
}
print("\nLive inspection captured. Exit 0 ≠ every capability works; read statuses above.")


## Section 6 — Hardware tests

Project marker: `hardware` (`pyproject.toml`). The ordinary suite **excludes** hardware tests (`addopts = … -m 'not hardware'`). Run them explicitly:

```bash
python -m pytest -m hardware tests/hardware -rA -p no:cacheprovider
```

Outcomes to expect and how to read them:

| Outcome | Meaning |
| ------- | ------- |
| PASSED | Preconditions held and assertions succeeded on this machine |
| FAILED | Preconditions held but an assertion failed — report it |
| SKIPPED | A precondition was missing (message says which); **not a pass** |
| BLOCKED | Used at the evidence-bundle / tracker level when NVIDIA hardware isn't available at all |

Optional tests stay **disabled** unless you set the configuration flags:

- `OPT_IN_CUDA_SMOKE` → evidence script `--cuda-smoke-test` (tiny CUDA compute in a child).
- `OPT_IN_HW12_ALLOCATION` → sets `GSI_HW_ALLOW_CUDA_ALLOCATION=1` for HW-12 (allocates GPU memory).

Neither feeds the inspector's diagnostic conclusions.



In [ ]:
import os
import subprocess
import sys
from pathlib import Path

HW_DIR = WORK_ROOT / "hardware-tests"
HW_DIR.mkdir(parents=True, exist_ok=True)
junit = HW_DIR / "hardware-tests.xml"
log_path = HW_DIR / "hardware-tests.log"

env = os.environ.copy()
if OPT_IN_HW12_ALLOCATION:
    env["GSI_HW_ALLOW_CUDA_ALLOCATION"] = "1"
    print("OPT-IN: GSI_HW_ALLOW_CUDA_ALLOCATION=1 (HW-12 may allocate GPU memory)")
else:
    env.pop("GSI_HW_ALLOW_CUDA_ALLOCATION", None)
    print("HW-12 allocation: disabled (default)")

cmd = [
    sys.executable,
    "-m",
    "pytest",
    "-m",
    "hardware",
    "tests/hardware",
    "-rA",
    "-p",
    "no:cacheprovider",
    "-o",
    "junit_family=xunit1",
    f"--junitxml={junit}",
]
print("$", " ".join(cmd))
result = subprocess.run(
    cmd,
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
    timeout=600,
    env=env,
)
log_path.write_text(result.stdout + "\n" + result.stderr)
print(result.stdout[-4000:] if len(result.stdout) > 4000 else result.stdout)
if result.stderr.strip():
    print("stderr:", result.stderr.strip()[:1000])
print("pytest exit:", result.returncode, "(non-zero can mean failures or collection errors)")

# Summarise junit outcomes without altering them
import xml.etree.ElementTree as ET

HW_RESULTS = []
if junit.exists():
    for case in ET.parse(junit).getroot().iter("testcase"):
        outcome, message = "passed", None
        for tag, name in (("failure", "failed"), ("error", "error"), ("skipped", "skipped")):
            el = case.find(tag)
            if el is not None:
                outcome, message = name, el.get("message")
                break
        HW_RESULTS.append({"test": case.get("name"), "outcome": outcome, "message": message})
    from collections import Counter
    counts = Counter(r["outcome"] for r in HW_RESULTS)
    print("\nOutcome counts:", dict(counts))
    for r in HW_RESULTS:
        msg = (r["message"] or "")[:120].replace("\n", " ")
        print(f"  {r['outcome']:7} {r['test']}" + (f" — {msg}" if msg else ""))
else:
    print("No junit XML written — see the log for details.")

print("\nReminder: SKIPPED is not PASSED. Do not re-run with weakened assertions.")


## Section 7 — Generate the official evidence bundle

Uses the project's `validation/hardware_evidence.py` unchanged. It writes:

- `SUMMARY.md`, `summary.json`
- `inspect-modeA.*`, `inspect-modeA-redacted.*`, `inspect-modeB.*`
- `hardware-tests.xml`, `hardware-tests.log`
- `environment.json`, `reference/`, optional `cuda-smoke-test.json`

Redacted Mode A output is included by default. Unredacted files still contain GPU UUIDs, PCI bus IDs and absolute paths — share those privately.

Optional flags (from the configuration cell):

- `--cuda-smoke-test` when `OPT_IN_CUDA_SMOKE` is True
- `--allow-cuda-allocation` when `OPT_IN_HW12_ALLOCATION` is True



In [ ]:
import json
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

if EVIDENCE_ROOT.exists():
    shutil.rmtree(EVIDENCE_ROOT)

cmd = [
    sys.executable,
    str(PROJECT_ROOT / "validation" / "hardware_evidence.py"),
    "--label",
    EVIDENCE_LABEL,
    "--out",
    str(EVIDENCE_ROOT),
    "--required-vram",
    str(REQUIRED_VRAM_GIB),
]
if OPT_IN_CUDA_SMOKE:
    cmd.append("--cuda-smoke-test")
    print("OPT-IN: --cuda-smoke-test (separate CUDA execution check)")
if OPT_IN_HW12_ALLOCATION:
    cmd.append("--allow-cuda-allocation")
    print("OPT-IN: --allow-cuda-allocation (HW-12)")

print("$", " ".join(cmd))
evidence = subprocess.run(cmd, cwd=PROJECT_ROOT, capture_output=True, text=True, timeout=900)
print(evidence.stdout)
if evidence.stderr.strip():
    print("stderr:", evidence.stderr.strip()[:2000])
print("evidence script exit:", evidence.returncode)

required = ["SUMMARY.md", "summary.json", "environment.json"]
missing = [n for n in required if not (EVIDENCE_ROOT / n).is_file()]
if missing:
    raise SystemExit(f"Evidence bundle incomplete; missing: {missing}")

summary = json.loads((EVIDENCE_ROOT / "summary.json").read_text())
hw = summary.get("hardware_tests") or {}
hw_results = hw.get("results") if isinstance(hw, dict) else None

provenance = {
    "notebook": "inferforge_nvidia_gpu_validation.ipynb",
    "validation_timestamp": datetime.now(timezone.utc).isoformat(),
    "source_mode": SOURCE.get("mode"),
    "source_repository": SOURCE.get("repository") or SOURCE.get("repo_url") or "unknown",
    "source_repo_url": SOURCE.get("repo_url") or "unknown",
    "source_branch": SOURCE.get("branch") if SOURCE.get("branch") is not None else "unknown",
    "source_commit": SOURCE.get("commit") if SOURCE.get("commit") is not None else "unknown",
    "source_commit_status": SOURCE.get("commit_status") or "unknown",
    "source_download_url": SOURCE.get("download_url") or "not_applicable",
    "source_archive_sha256": SOURCE.get("archive_sha256") or "unknown",
    "gpu_model": RUNTIME.get("gpu_name") or "unknown",
    "gpu_count": RUNTIME.get("gpu_count"),
    "driver_version": RUNTIME.get("driver_version") or "unknown",
    "cuda_environment": RUNTIME.get("cuda_env") or {},
    "runtime_python": RUNTIME.get("python_version") or "unknown",
    "runtime_platform": RUNTIME.get("platform") or "unknown",
    "evidence_verdict": summary.get("verdict") or "unknown",
    "evidence_script_exit": evidence.returncode,
    "hardware_test_results": hw_results if hw_results is not None else [],
    "hardware_tests_error": hw.get("error") if isinstance(hw, dict) else None,
    "install": INSTALL,
    "source": SOURCE,
    "runtime": {
        k: RUNTIME.get(k)
        for k in (
            "collected_at",
            "python_version",
            "platform",
            "machine",
            "in_colab",
            "gpu_name",
            "driver_version",
            "gpu_count",
            "verdict",
            "cuda_env",
        )
    },
}
(EVIDENCE_ROOT / "notebook-run-meta.json").write_text(json.dumps(provenance, indent=2) + "\n")

# Human-readable provenance (does not alter original test result files)
prov_lines = [
    "# Notebook run provenance",
    "",
    f"- Validation timestamp: {provenance['validation_timestamp']}",
    f"- Source mode: {provenance['source_mode']}",
    f"- Repository: {provenance['source_repository']}",
    f"- Repo URL: {provenance['source_repo_url']}",
    f"- Branch: {provenance['source_branch']}",
    f"- Commit SHA: {provenance['source_commit']} ({provenance['source_commit_status']})",
    f"- Download URL: {provenance['source_download_url']}",
    f"- Source archive sha256: {provenance['source_archive_sha256']}",
    f"- GPU model: {provenance['gpu_model']}",
    f"- GPU count: {provenance['gpu_count']}",
    f"- Driver version: {provenance['driver_version']}",
    f"- CUDA-related env: {json.dumps(provenance['cuda_environment'])}",
    f"- Evidence verdict: {provenance['evidence_verdict']}",
    f"- Evidence script exit: {provenance['evidence_script_exit']}",
    "",
    "Unknown values are labelled explicitly; nothing above is invented.",
    "",
    "## Hardware test outcomes (from evidence bundle)",
    "",
]
if provenance["hardware_tests_error"]:
    prov_lines.append(f"Not fully listed: {provenance['hardware_tests_error']}")
elif not provenance["hardware_test_results"]:
    prov_lines.append("No hardware test results recorded.")
else:
    for row in provenance["hardware_test_results"]:
        msg = (row.get("message") or "").replace("\n", " ")[:160]
        prov_lines.append(
            f"- {row.get('outcome', 'unknown')}: {row.get('test', 'unknown')}"
            + (f" — {msg}" if msg else "")
        )
prov_lines.append("")
(EVIDENCE_ROOT / "PROVENANCE.md").write_text("\n".join(prov_lines))

# Append a short pointer to SUMMARY.md without rewriting its verdict logic
summary_text = (EVIDENCE_ROOT / "SUMMARY.md").read_text()
if "## Notebook provenance" not in summary_text:
    summary_text = (
        summary_text.rstrip()
        + "\n\n## Notebook provenance\n\n"
        + f"- Source mode: `{provenance['source_mode']}`\n"
        + f"- Repository: `{provenance['source_repository']}`\n"
        + f"- Branch: `{provenance['source_branch']}`\n"
        + f"- Commit: `{provenance['source_commit']}` "
        + f"({provenance['source_commit_status']})\n"
        + f"- GPU model: `{provenance['gpu_model']}`\n"
        + f"- Driver: `{provenance['driver_version']}`\n"
        + "- Details: `PROVENANCE.md`, `notebook-run-meta.json`\n"
    )
    (EVIDENCE_ROOT / "SUMMARY.md").write_text(summary_text)

print("\n=== SUMMARY.md (head) ===")
print("\n".join((EVIDENCE_ROOT / "SUMMARY.md").read_text().splitlines()[:45]))
print("\nVerdict:", summary.get("verdict"))
print("Provenance commit:", provenance["source_commit"], provenance["source_commit_status"])
print("Bundle:", EVIDENCE_ROOT)
print("Contents:", sorted(p.name for p in EVIDENCE_ROOT.iterdir()))


## Section 8 — Download the evidence archive

Creates `gsi-evidence.tar.gz` in `/content` and offers a Colab download when available.

**Sharing:** attach the archive privately to a GitHub issue, or attach at least `SUMMARY.md`, `summary.json` and `hardware-tests.xml` plus a note of who ran it, when, and on which GPU. See `docs/hardware_validation.md` §5.



In [ ]:
import json
import tarfile
from datetime import datetime, timezone
from pathlib import Path

ARCHIVE_OUT = Path("/content/gsi-evidence.tar.gz")
if not EVIDENCE_ROOT.is_dir() or not (EVIDENCE_ROOT / "SUMMARY.md").is_file():
    raise SystemExit("Evidence directory missing — run Section 7 first.")

if ARCHIVE_OUT.exists():
    ARCHIVE_OUT.unlink()

with tarfile.open(ARCHIVE_OUT, "w:gz") as tf:
    tf.add(EVIDENCE_ROOT, arcname=EVIDENCE_ROOT.name)

assert ARCHIVE_OUT.is_file() and ARCHIVE_OUT.stat().st_size > 0, "Archive was not created"

summary = json.loads((EVIDENCE_ROOT / "summary.json").read_text())
meta_path = EVIDENCE_ROOT / "notebook-run-meta.json"
meta = json.loads(meta_path.read_text()) if meta_path.is_file() else {}

DOWNLOAD = {
    "archive": str(ARCHIVE_OUT),
    "bytes": ARCHIVE_OUT.stat().st_size,
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "gpu_model": meta.get("gpu_model") or RUNTIME.get("gpu_name") or "unknown",
    "driver_version": meta.get("driver_version") or RUNTIME.get("driver_version") or "unknown",
    "source_mode": meta.get("source_mode") or SOURCE.get("mode") or "unknown",
    "source_repository": meta.get("source_repository") or "unknown",
    "source_branch": meta.get("source_branch") or "unknown",
    "source_commit": meta.get("source_commit") or "unknown",
    "overall_result": summary.get("verdict") or "unknown",
    "evidence_script_exit": meta.get("evidence_script_exit"),
}

print("Archive:", DOWNLOAD["archive"])
print("Size:", DOWNLOAD["bytes"], "bytes")
print("Timestamp:", DOWNLOAD["timestamp"])
print("GPU model:", DOWNLOAD["gpu_model"])
print("Driver:", DOWNLOAD["driver_version"])
print("Source mode:", DOWNLOAD["source_mode"])
print("Repository:", DOWNLOAD["source_repository"])
print("Branch:", DOWNLOAD["source_branch"])
print("Commit:", DOWNLOAD["source_commit"])
print("Overall result:", DOWNLOAD["overall_result"])

try:
    from google.colab import files

    files.download(str(ARCHIVE_OUT))
    print("Colab download started.")
except Exception as exc:  # noqa: BLE001
    print(
        f"Colab download helper unavailable ({type(exc).__name__}: {exc}). "
        f"Download manually from {ARCHIVE_OUT}."
    )


## Section 9 — Interpretation (do not go beyond the evidence)

Use `SUMMARY.md` and the live report. The classifications below come from the evidence script.

| Observation | How to read it |
| ----------- | -------------- |
| NVIDIA devices `COLLECTED` with names/UUIDs | Successful detection via NVML (or the documented `nvidia-smi` fallback). Compare with the independent `reference/nvidia-smi-*.json` outputs. |
| Driver / toolkit / PyTorch CUDA versions | Installation **facts**, kept in separate fields. They are **not** proof that CUDA code runs. D03's legend spells this out. |
| NVML `NOT_DETECTED` / library not found | Driver user-mode library not loadable here. On Colab with a GPU runtime this is unexpected — note the runtime type and any `pynvml` conflict. |
| PyTorch `cuda_available` | PyTorch's own answer from a child probe. Ordinary inspection runs **no** CUDA code. |
| Correlation `NOT_DETECTED` / unresolved | UUID then PCI matching; ambiguity stays unresolved (never a wrong match). Wide PCI domains are an open owner decision (KI-38). |
| Feature `NOT_SUPPORTED_ON_THIS_ENVIRONMENT` | Platform/driver limitation (e.g. some metrics on WSL2), not a test failure. |
| Feature `NOT_IMPLEMENTED` | Inspector gap (e.g. graphics processes KI-36) — absence ≠ "not on the machine". |
| Hardware test SKIPPED | Precondition missing; **not a pass**. |
| Hardware test FAILED | Real mismatch with the pass criteria — file a bug with the redacted report. |
| Evidence verdict `EVIDENCE_COLLECTED` | Bundle ready for **owner review**; not yet `HARDWARE_VALIDATED`. |
| Evidence verdict `BLOCKED` | No NVIDIA GPU accessible — this notebook should have stopped in Section 2. |

Do not claim a successful NVIDIA validation from this notebook alone until the owner records the HW items from your evidence.



## Section 10 — Optional cleanup

Removes only the temporary checkout / extract tree and intermediate live-inspect copies under `/content/inferforge-validation`.

**Not deleted** unless you uncomment the lines yourself:

- your uploaded source archive (`ARCHIVE_PATH`)
- the evidence directory and `gsi-evidence.tar.gz` download



In [ ]:
import shutil
from pathlib import Path

# Safe default: only temporary working files.
if WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)
    print("Removed:", WORK_ROOT)
else:
    print("Nothing to remove at", WORK_ROOT)

# Uncomment ONLY if you intentionally want to delete evidence / archive too:
# if EVIDENCE_ROOT.exists():
#     shutil.rmtree(EVIDENCE_ROOT)
#     print("Removed evidence dir:", EVIDENCE_ROOT)
# archive = Path("/content/gsi-evidence.tar.gz")
# if archive.exists():
#     archive.unlink()
#     print("Removed", archive)
# if SOURCE_MODE == "archive":
#     ap = Path(ARCHIVE_PATH)
#     if ap.exists():
#         ap.unlink()
#         print("Removed uploaded archive", ap)

print("Cleanup done. Re-run from Section 2 for another validation.")
